# Final Project Experiments

In [1]:
import cv2 as cv
import os
import numpy as np

INPUT_FILEPATH = "data/input/"
OUTPUT_FILEPATH = "data/output/"

RAND_SEED = 6524
np.random.seed(RAND_SEED)

NOISE_GAMMA = 3.0
NOISE_SIGMAS = [0.025, 0.05, 0.075, 0.1, 0.125]
DARKEN_GAMMAS = [3.0, 3.5, 4.0, 4.5, 5.0]

In [2]:
def add_noise(image, sigma):
    # Rescale image to [0, 1]
    rescaled_image = image / 255

    # Generate Gaussian noise
    noise = np.random.normal(0, sigma, (image.shape[0], image.shape[1], image.shape[2]))

    noisy_image = rescaled_image + noise                    # Add noise
    noisy_image = noisy_image * 255.0                       # Scale image back to [0, 255]
    noisy_image = np.clip(noisy_image, 0, 255)              # Clip to range [0, 255]
    noisy_image = np.round(noisy_image).astype(np.uint8)    # Convert to integers
    
    return noisy_image

def gamma_adjust(image, gamma):

    # Convert image to HSV
    hsv_image = cv.cvtColor(image, cv.COLOR_BGR2HSV)
    
    # Scale intensity values to [0, 1]
    scaled_intensity = hsv_image[:, :, 2] / 255

    gamma_intensity = np.power(scaled_intensity, gamma)           # Apply gamma correction

    gamma_intensity = gamma_intensity * 255.0                       # Scale image back to [0, 255]
    gamma_intensity = np.clip(gamma_intensity, 0, 255)              # Clip to range [0, 255]
    gamma_intensity = np.round(gamma_intensity).astype(np.uint8)    # Convert to integers

    hsv_image[:, :, 2] = gamma_intensity

    output = cv.cvtColor(hsv_image, cv.COLOR_HSV2BGR)

    return output

def noise_experiments(image, name, ext):

    for sigma in NOISE_SIGMAS:
        
        gamma_image = gamma_adjust(image, NOISE_GAMMA)  # Apply gamma correction to darken image
        noisy_image = add_noise(gamma_image, sigma)     # Add noise

        # Ouput with corresponding name
        output_name = "{}_noise_sigma_{}.{}".format(name, str(sigma).replace(".", "-"), ext)
        cv.imwrite(OUTPUT_FILEPATH + output_name, noisy_image)

# Gamma values to be tested
def gamma_experiments(image, name, ext):

    for gam in DARKEN_GAMMAS:
        gamma_image = gamma_adjust(image, gam)  # Apply gamma correction to darken image

        # Ouput with corresponding name
        output_name = "{}_gamma_{}.{}".format(name, str(gam).replace(".", "-"), ext)
        cv.imwrite(OUTPUT_FILEPATH + output_name, gamma_image)

In [3]:
# Remove any existing files
existing_files = os.listdir(OUTPUT_FILEPATH)
existing_files.remove(".gitignore") # Don't remove .gitignore

for file in existing_files:
    os.remove(OUTPUT_FILEPATH + file)

# Processing images
filenames = os.listdir(INPUT_FILEPATH)
filenames.remove(".gitignore") # Don't try to process .gitignore as an image

for name_file in filenames:

    name, ext = name_file.split(".")

    image = cv.imread(INPUT_FILEPATH + name_file)

    noise_experiments(image, name, ext)
    gamma_experiments(image, name, ext)